# Audiobook Enhancer — Colab Bootstrap

This notebook is a bootstrapper only. It never contains pipeline logic —
all processing code lives in the `audiobook_engine` package in the repo.
Edit code there, not here. See `02-Architecture.md` in the vault.

In [ ]:
# 1. Clone or pull the repo (absolute path — avoids nested-clone bugs from
# a lingering %cd in another cell's execution state)
REPO_DIR = "/content/audiobook-enhancer"
REPO_URL = "https://github.com/hasnaintypes/audiobook-enhancer.git"
import os
if os.path.isdir(REPO_DIR):
    !cd "$REPO_DIR" && git pull
else:
    !git clone "$REPO_URL" "$REPO_DIR"

In [ ]:
# 2. Install pinned dependencies
import os
os.chdir("/content/audiobook-enhancer")
!pip install -q -r config/requirements.lock
!pip install -q -e .

In [ ]:
# 3. Load Neon Object Storage credentials from Colab's Secrets manager
# (the key icon in the left sidebar) — NOT pasted into this cell. Add each
# of these as a secret there first, with "Notebook access" enabled:
#   AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_ENDPOINT_URL_S3, AWS_REGION
# Values come from audiobook-enhancer/.env.local on your own machine
# (written there by `neon deploy` — see docs/07-Decisions.md D007).
import os
from google.colab import userdata

for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_ENDPOINT_URL_S3", "AWS_REGION"]:
    os.environ[key] = userdata.get(key)

print("Neon Object Storage credentials loaded from Colab Secrets.")

In [ ]:
# 4. Sanity check: run the fast unit test suite here to confirm the clone +
# dependency install actually produced a working environment.
!pytest tests/unit -q

In [ ]:
# 5. One-off smoke test: run a short clip from a test video through the
# pipeline (extract -> analyze -> upload) to confirm the environment works
# end-to-end. Not a run against the real sample set -- swap URLS for that.
#
# extract_audio() in the engine always pulls the full video, so for a quick
# test here we download only a CLIP_SECONDS-long window via yt-dlp's
# download_ranges instead of calling extract_audio() directly -- this is
# diagnostic-script logic for this cell only, not a change to the engine.
# The window is centered on the video's midpoint rather than the start.
script = '''
from pathlib import Path
import yt_dlp
from audiobook_engine.analysis.io import load_mono
from audiobook_engine.analysis.quality import analyze_quality
from audiobook_engine.analysis.strategy import select_strategy
from audiobook_engine.jobstore.storage import upload_file, object_key, presigned_url

URLS = [
    "https://youtu.be/XN9CVvZFkts",
]
CLIP_SECONDS = 60
OUTPUT_DIR = Path("/content/samples")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def extract_middle_clip(url, output_dir, clip_seconds):
    probe_opts = {"quiet": True, "no_warnings": True, "noplaylist": True}
    with yt_dlp.YoutubeDL(probe_opts) as ydl:
        probe_info = ydl.extract_info(url, download=False)
    duration = float(probe_info.get("duration") or 0.0)

    midpoint = duration / 2
    start = max(0.0, midpoint - clip_seconds / 2)
    end = min(duration, start + clip_seconds) if duration else start + clip_seconds

    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": str(output_dir / "%(id)s.%(ext)s"),
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav"}],
        "download_ranges": yt_dlp.utils.download_range_func(None, [(start, end)]),
        "force_keyframes_at_cuts": True,
        "noplaylist": True,
        "quiet": True,
        "no_warnings": True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
    video_id = info["id"]
    audio_path = output_dir / f"{video_id}.wav"
    return audio_path, info.get("title", "unknown"), duration, start, end

for i, url in enumerate(URLS, start=1):
    job_id = f"sample-{i:03d}"
    print("=" * 80)
    print(url, "->", job_id, f"({CLIP_SECONDS}s clip from the middle)")
    try:
        audio_path, title, duration, start, end = extract_middle_clip(url, OUTPUT_DIR, CLIP_SECONDS)
        print(f"Title: {title}")
        print(f"Full video duration: {duration:.1f}s -- clipped [{start:.1f}s, {end:.1f}s]")

        samples, sr = load_mono(audio_path)
        report = analyze_quality(samples, sr)
        decision = select_strategy(report)
        print(report)
        print(decision)

        key = object_key(job_id, "original", audio_path.name)
        upload_file(audio_path, key)
        print(f"Uploaded to Neon Object Storage: {key}")

        url_out = presigned_url(key, expires_in=600)
        print(f"Presigned URL (10 min): {url_out[:80]}...")
    except Exception as e:
        print(f"Failed: {e}")
'''
with open("/content/smoke_test_clip.py", "w") as f:
    f.write(script)

!python /content/smoke_test_clip.py

In [ ]:
# 6. Check sample-004 against the new detectors (background music,
# reverb, compression artifacts) -- this sample came back "clean" on
# the old metrics despite sounding bad by ear.
script = '''
from pathlib import Path
from audiobook_engine.ingestion.youtube import extract_audio, ExtractionError
from audiobook_engine.analysis.io import load_mono
from audiobook_engine.analysis.quality import analyze_quality

URL = "https://youtu.be/mJf-Wm1iJBY"  # sample-004, "clean" under the old metrics
OUTPUT_DIR = Path("/content/samples")

try:
    extracted = extract_audio(URL, OUTPUT_DIR)
    print(f"Title: {extracted.title}")

    samples, sr = load_mono(extracted.audio_path)
    report = analyze_quality(samples, sr)

    print(f"has_background_music:       {report.has_background_music} (confidence={report.background_music_confidence:.3f})")
    print(f"has_reverb:                 {report.has_reverb} (tail_score={report.reverb_tail_score:.3f})")
    print(f"has_compression_artifacts:  {report.has_compression_artifacts} (spectral_cutoff_hz={report.spectral_cutoff_hz:.1f})")
except ExtractionError as e:
    print(f"Extraction failed [{e.kind.value}]: {e}")
'''
with open("/content/check_sample_004.py", "w") as f:
    f.write(script)

!python /content/check_sample_004.py

In [ ]:
# 7. Stress-test RNNoise's -ar flag for the same end-of-stream hang that
# hit Tier 0's combined filter+resample ffmpeg call. Only the 2-minute
# clips tested so far never triggered it -- this uses a real 10-minute
# clip to push harder on the same code path, with a timeout so a real
# hang shows up as a TimeoutExpired instead of blocking the kernel.
script = """
import subprocess
import time
from pathlib import Path
import yt_dlp

from audiobook_engine.enhancement.rnnoise import download_default_model, build_ffmpeg_command

URL = "https://youtu.be/XN9CVvZFkts"
CLIP_SECONDS = 600  # 10 minutes -- stress test, not the earlier 60s smoke test
OUTPUT_DIR = Path("/content/samples")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = Path("/content/models/std.rnnn")

model_path = download_default_model(MODEL_PATH)
print(f"Model ready at {model_path}")

probe_opts = {"quiet": True, "no_warnings": True, "noplaylist": True}
with yt_dlp.YoutubeDL(probe_opts) as ydl:
    probe_info = ydl.extract_info(URL, download=False)
duration = float(probe_info.get("duration") or 0.0)
midpoint = duration / 2
start = max(0.0, midpoint - CLIP_SECONDS / 2)
end = min(duration, start + CLIP_SECONDS)
print(f"Full duration: {duration:.1f}s -- stress clip [{start:.1f}s, {end:.1f}s]")

ydl_opts = {
    "format": "bestaudio/best",
    "outtmpl": str(OUTPUT_DIR / "%(id)s_stress.%(ext)s"),
    "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav"}],
    "download_ranges": yt_dlp.utils.download_range_func(None, [(start, end)]),
    "force_keyframes_at_cuts": True,
    "noplaylist": True,
    "quiet": True,
    "no_warnings": True,
}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(URL, download=True)
input_path = OUTPUT_DIR / f"{info['id']}_stress.wav"
output_path = OUTPUT_DIR / f"{info['id']}_stress_denoised.wav"
print(f"Downloaded clip: {input_path} ({input_path.stat().st_size} bytes)")

command = build_ffmpeg_command(input_path, output_path, model_path, sample_rate=48000)
print("Running:", " ".join(command))

start_time = time.monotonic()
try:
    result = subprocess.run(command, check=True, capture_output=True, text=True, timeout=120)
    elapsed = time.monotonic() - start_time
    print(f"Completed in {elapsed:.1f}s -- no hang. Output: {output_path} ({output_path.stat().st_size} bytes)")
except subprocess.TimeoutExpired:
    elapsed = time.monotonic() - start_time
    print(f"HUNG: did not complete within {elapsed:.1f}s timeout -- reproduces the Tier 0-style hang")
except subprocess.CalledProcessError as e:
    print(f"ffmpeg failed (exit {e.returncode}): {e.stderr[-2000:]}")
"""
with open("/content/stress_test_rnnoise.py", "w") as f:
    f.write(script)

!python /content/stress_test_rnnoise.py

In [ ]:
# 8. Run the new per-chunk job runner (audiobook_engine.runner.job_runner)
# end-to-end against a real long sample -- long enough (>600s, the default
# ChunkingConfig.max_chunk_duration_s) to actually produce multiple chunks,
# not just exercise the single-chunk "end_of_audio" path.
script = """
from pathlib import Path
import yt_dlp

from audiobook_engine.runner.job_runner import run_job

URL = "https://youtu.be/XN9CVvZFkts"
CLIP_SECONDS = 1200  # 20 minutes -- long enough to force at least 2-3 chunks
OUTPUT_DIR = Path("/content/samples")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path("/content/job_work")
JOB_ID = "runner-smoke-001"

probe_opts = {"quiet": True, "no_warnings": True, "noplaylist": True}
with yt_dlp.YoutubeDL(probe_opts) as ydl:
    probe_info = ydl.extract_info(URL, download=False)
duration = float(probe_info.get("duration") or 0.0)
start = 0.0
end = min(duration, CLIP_SECONDS)
print(f"Full duration: {duration:.1f}s -- runner smoke clip [{start:.1f}s, {end:.1f}s]")

ydl_opts = {
    "format": "bestaudio/best",
    "outtmpl": str(OUTPUT_DIR / "%(id)s_runner.%(ext)s"),
    "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav"}],
    "download_ranges": yt_dlp.utils.download_range_func(None, [(start, end)]),
    "force_keyframes_at_cuts": True,
    "noplaylist": True,
    "quiet": True,
    "no_warnings": True,
}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(URL, download=True)
input_path = OUTPUT_DIR / f"{info['id']}_runner.wav"
print(f"Downloaded clip: {input_path} ({input_path.stat().st_size} bytes)")

manifest = run_job(JOB_ID, input_path, WORK_DIR)
print(f"Job {manifest.job_id}: {len(manifest.completed_indices)}/{manifest.total_chunks} chunks completed")
print(f"Completed indices: {sorted(manifest.completed_indices)}")
"""
with open("/content/run_job_smoke_test.py", "w") as f:
    f.write(script)

!python /content/run_job_smoke_test.py

In [ ]:
# 9. Real deliberate-interruption test for the job runner (docs/06-Roadmap.md
# Phase 4's and the whole PRD's stated success metric): simulate the process
# dying mid-job via a precisely-timed fault injection on a real clip, confirm
# the persisted manifest reflects only truly-completed chunks, then resume
# the SAME job_id and confirm completed chunks are never reprocessed while
# the chunk that crashed before its completion was persisted is safely redone.
script = """
from pathlib import Path
from collections import Counter
import yt_dlp

import audiobook_engine.jobstore.storage as storage_module
import audiobook_engine.postprocess.dsp as dsp_module
import audiobook_engine.enhancement.rnnoise as rnnoise_module
from audiobook_engine.runner.job_runner import run_job

URL = "https://youtu.be/XN9CVvZFkts"
CLIP_SECONDS = 1800  # 30 minutes -- long enough for several chunks at the 600s default
OUTPUT_DIR = Path("/content/samples")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path("/content/job_work_interrupt")
JOB_ID = "interruption-test-001"

probe_opts = {"quiet": True, "no_warnings": True, "noplaylist": True}
with yt_dlp.YoutubeDL(probe_opts) as ydl:
    probe_info = ydl.extract_info(URL, download=False)
duration = float(probe_info.get("duration") or 0.0)
start = 1200.0  # a different window than the earlier smoke-test clip
end = min(duration, start + CLIP_SECONDS)
print(f"Full duration: {duration:.1f}s -- interruption-test clip [{start:.1f}s, {end:.1f}s]")

ydl_opts = {
    "format": "bestaudio/best",
    "outtmpl": str(OUTPUT_DIR / "%(id)s_interrupt.%(ext)s"),
    "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav"}],
    "download_ranges": yt_dlp.utils.download_range_func(None, [(start, end)]),
    "force_keyframes_at_cuts": True,
    "noplaylist": True,
    "quiet": True,
    "no_warnings": True,
}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(URL, download=True)
input_path = OUTPUT_DIR / f"{info['id']}_interrupt.wav"
print(f"Downloaded clip: {input_path} ({input_path.stat().st_size} bytes)")

# Instrumentation: log every real chunk-processing call by its input filename,
# across BOTH the crashed run and the resumed run, so we can prove afterward
# which chunks were processed exactly once (correctly not reprocessed) vs.
# processed twice (correctly redone, because their completion was never
# persisted before the simulated crash).
processing_log = []
real_tier0 = dsp_module.run_tier0_processing
real_tier1 = rnnoise_module.run_tier1_processing

def logging_tier0(input_path, output_path, decision, sample_rate):
    processing_log.append(input_path.name)
    return real_tier0(input_path, output_path, decision, sample_rate)

def logging_tier1(input_path, output_path, model_path, sample_rate):
    processing_log.append(input_path.name)
    return real_tier1(input_path, output_path, model_path, sample_rate)

dsp_module.run_tier0_processing = logging_tier0
rnnoise_module.run_tier1_processing = logging_tier1

# Simulate a real crash: let the 2nd upload_manifest call succeed (chunk 0's
# completion durably persisted), then make the 3rd call raise instead of
# persisting -- simulating the process dying while chunk 1 is being
# finished, BEFORE its completion checkpoint is written.
real_upload_manifest = storage_module.upload_manifest
call_count = {"n": 0}

def crashing_upload_manifest(manifest):
    call_count["n"] += 1
    if call_count["n"] == 3:
        raise RuntimeError("SIMULATED INTERRUPTION -- process killed before this checkpoint")
    return real_upload_manifest(manifest)

storage_module.upload_manifest = crashing_upload_manifest

print("=" * 80)
print("PHASE 1: run_job with a simulated crash partway through")
try:
    run_job(JOB_ID, input_path, WORK_DIR)
    print("UNEXPECTED: run_job completed without the simulated crash firing")
except RuntimeError as e:
    print(f"Job crashed as expected: {e}")

storage_module.upload_manifest = real_upload_manifest  # restore before checking/resuming

manifest_after_crash = storage_module.download_manifest(JOB_ID)
print(f"Manifest after crash: {sorted(manifest_after_crash.completed_indices)} of {manifest_after_crash.total_chunks} persisted complete")
print(f"Chunks processed before crash: {processing_log}")

print("=" * 80)
print("PHASE 2: resume the SAME job_id with no fault injection")
final_manifest = run_job(JOB_ID, input_path, WORK_DIR)
print(f"Final manifest: {sorted(final_manifest.completed_indices)} of {final_manifest.total_chunks} complete")
print(f"All chunks processed (combined log across both runs): {processing_log}")

counts = Counter(processing_log)
once_only = sorted(name for name, n in counts.items() if n == 1)
reprocessed = sorted(name for name, n in counts.items() if n > 1)
print(f"Processed exactly once (correctly NOT reprocessed): {once_only}")
print(f"Processed more than once (correctly redone -- crashed before their completion was persisted): {reprocessed}")
"""
with open("/content/interruption_test.py", "w") as f:
    f.write(script)

!python /content/interruption_test.py

In [ ]:
# 11. Run the new crossfade stitching (audiobook_engine.stitching.stitch)
# against the already-completed runner-smoke-001 job, confirm the uploaded
# whole-file deliverable exists with the expected (crossfade-adjusted)
# duration -- mechanical verification only (duration/existence), not a
# by-ear "no audible seams" check, which is explicitly deferred.
script = """
from pathlib import Path
import soundfile as sf

from audiobook_engine.jobstore.storage import get_s3_client, BUCKET_NAME, object_key
from audiobook_engine.stitching.stitch import stitch_job

JOB_ID = "runner-smoke-001"
WORK_DIR = Path("/content/stitch_work")

stitched_path = stitch_job(JOB_ID, WORK_DIR, crossfade_ms=50.0)
samples, sr = sf.read(str(stitched_path))
duration_s = len(samples) / sr
print(f"Stitched file: {stitched_path} -- {duration_s:.2f}s at {sr}Hz")

client = get_s3_client()
key = object_key(JOB_ID, "enhanced", "final.wav")
head = client.head_object(Bucket=BUCKET_NAME, Key=key)
print(f"Uploaded deliverable confirmed in storage: {key} ({head['ContentLength']} bytes)")
"""
with open("/content/stitch_smoke_test.py", "w") as f:
    f.write(script)

!python /content/stitch_smoke_test.py

In [ ]:
# 12. ChunkingConfig default-tuning experiment (gap.md Step 6's last item):
# settle the three provisional defaults (max_chunk_duration_s=600.0,
# silence_rms_threshold_db=-45.0, min_silence_duration_s=0.5) against real,
# full-length samples instead of leaving them as unvalidated placeholders --
# same evidence-based process used to (eventually) validate quality.py's
# thresholds. This is a read-only data-gathering experiment; it does not
# change ChunkingConfig itself.
script = """
from pathlib import Path
import numpy as np
import yt_dlp

from audiobook_engine.analysis.io import load_mono
from audiobook_engine.chunking.boundaries import ChunkingConfig, find_silence_gaps, plan_chunks

# Two real full-length samples already used elsewhere in this notebook --
# different speakers/content (Tafseer lecture vs. Urdu novel narration).
SAMPLES = [
    ("https://youtu.be/XN9CVvZFkts", "tafseer-lecture"),
    ("https://youtu.be/mJf-Wm1iJBY", "urdu-novel"),
]
OUTPUT_DIR = Path("/content/samples")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def download_full(url, label):
    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": str(OUTPUT_DIR / f"{label}_full.%(ext)s"),
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav"}],
        "noplaylist": True,
        "quiet": True,
        "no_warnings": True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.extract_info(url, download=True)
    return OUTPUT_DIR / f"{label}_full.wav"

CANDIDATE_THRESHOLDS_DB = [-40.0, -45.0, -50.0]
CANDIDATE_MIN_SILENCE_S = [0.3, 0.5, 0.8]

for url, label in SAMPLES:
    print("=" * 80)
    print(f"Sample: {label} ({url})")
    audio_path = download_full(url, label)
    samples, sr = load_mono(audio_path)
    duration_s = len(samples) / sr
    print(f"  Full duration: {duration_s:.1f}s ({duration_s/60:.1f} min)")

    for threshold_db in CANDIDATE_THRESHOLDS_DB:
        for min_silence_s in CANDIDATE_MIN_SILENCE_S:
            config = ChunkingConfig(
                silence_rms_threshold_db=threshold_db,
                min_silence_duration_s=min_silence_s,
            )
            gaps = find_silence_gaps(samples, sr, config)
            gap_durations = [end - start for start, end in gaps]
            boundaries = plan_chunks(samples, sr, config)
            fallback_count = sum(1 for b in boundaries if b.cut_reason == "max_duration_fallback")
            silence_count = sum(1 for b in boundaries if b.cut_reason == "silence_gap")
            print(
                f"  threshold={threshold_db:>6.1f}dB min_silence={min_silence_s:>4.1f}s -- "
                f"{len(gaps):>4d} gaps found (avg {np.mean(gap_durations) if gap_durations else 0:.2f}s), "
                f"{len(boundaries):>3d} chunks ({silence_count} silence_gap, {fallback_count} max_duration_fallback)"
            )
"""
with open("/content/chunking_config_experiment.py", "w") as f:
    f.write(script)

!python /content/chunking_config_experiment.py

In [ ]:
# 13. Follow-up: the lecture sample found almost no silence gaps at any of
# the 3 tested thresholds. Check its actual noise floor (quality.py already
# computes this per-sample) and scan a much wider/more lenient threshold
# range to see whether a fixed global threshold can ever work for this kind
# of continuous-speech content, or whether the gate needs to be adaptive to
# each sample's own noise floor instead of a single global dB constant.
script = """
from pathlib import Path
import numpy as np

from audiobook_engine.analysis.io import load_mono
from audiobook_engine.analysis.quality import analyze_quality
from audiobook_engine.analysis.framing import frame_signal, rms_db
from audiobook_engine.chunking.boundaries import ChunkingConfig, find_silence_gaps

OUTPUT_DIR = Path("/content/samples")
SAMPLES = [
    (OUTPUT_DIR / "tafseer-lecture_full.wav", "tafseer-lecture"),
    (OUTPUT_DIR / "urdu-novel_full.wav", "urdu-novel"),
]

WIDE_THRESHOLDS_DB = [-45.0, -40.0, -35.0, -30.0, -25.0, -20.0]

for audio_path, label in SAMPLES:
    print("=" * 80)
    print(f"Sample: {label}")
    samples, sr = load_mono(audio_path)

    # Reuse quality.py's own frame-level RMS to show the actual distribution,
    # not just a pass/fail against one threshold.
    frames = frame_signal(samples, sr, frame_ms=20.0)
    frame_dbs = np.array([rms_db(f) for f in frames])
    print(
        f"  Frame RMS (dB) distribution: min={frame_dbs.min():.1f} "
        f"p10={np.percentile(frame_dbs, 10):.1f} p50={np.percentile(frame_dbs, 50):.1f} "
        f"p90={np.percentile(frame_dbs, 90):.1f} max={frame_dbs.max():.1f}"
    )

    report = analyze_quality(samples, sr)
    print(f"  analyze_quality noise_floor_db: {report.noise_floor_db:.1f}  silence_fraction: {report.silence_fraction:.4f}")

    for threshold_db in WIDE_THRESHOLDS_DB:
        config = ChunkingConfig(silence_rms_threshold_db=threshold_db, min_silence_duration_s=0.5)
        gaps = find_silence_gaps(samples, sr, config)
        print(f"  threshold={threshold_db:>6.1f}dB -- {len(gaps):>4d} gaps found")
"""
with open("/content/chunking_threshold_scan.py", "w") as f:
    f.write(script)

!python /content/chunking_threshold_scan.py

In [ ]:
# 14. Verify the fix: re-plan chunks for both real samples using the new
# adaptive threshold (estimate_adaptive_silence_threshold_db) instead of a
# fixed global dB value, and confirm the lecture sample's max_duration_fallback
# count actually drops while the novel sample doesn't regress.
script = """
from pathlib import Path
import dataclasses

from audiobook_engine.analysis.io import load_mono
from audiobook_engine.chunking.boundaries import (
    ChunkingConfig,
    estimate_adaptive_silence_threshold_db,
    plan_chunks,
)

OUTPUT_DIR = Path("/content/samples")
SAMPLES = [
    (OUTPUT_DIR / "tafseer-lecture_full.wav", "tafseer-lecture"),
    (OUTPUT_DIR / "urdu-novel_full.wav", "urdu-novel"),
]

for audio_path, label in SAMPLES:
    print("=" * 80)
    print(f"Sample: {label}")
    samples, sr = load_mono(audio_path)

    base_config = ChunkingConfig()
    adaptive_threshold = estimate_adaptive_silence_threshold_db(samples, sr, frame_ms=base_config.frame_ms)
    effective_config = dataclasses.replace(base_config, silence_rms_threshold_db=adaptive_threshold)
    print(f"  Adaptive threshold: {adaptive_threshold:.1f}dB (old fixed default was {base_config.silence_rms_threshold_db:.1f}dB)")

    old_boundaries = plan_chunks(samples, sr, base_config)
    new_boundaries = plan_chunks(samples, sr, effective_config)

    old_fallback = sum(1 for b in old_boundaries if b.cut_reason == "max_duration_fallback")
    new_fallback = sum(1 for b in new_boundaries if b.cut_reason == "max_duration_fallback")
    old_silence = sum(1 for b in old_boundaries if b.cut_reason == "silence_gap")
    new_silence = sum(1 for b in new_boundaries if b.cut_reason == "silence_gap")

    print(f"  OLD (fixed -45dB):      {len(old_boundaries)} chunks -- {old_silence} silence_gap, {old_fallback} max_duration_fallback")
    print(f"  NEW (adaptive):         {len(new_boundaries)} chunks -- {new_silence} silence_gap, {new_fallback} max_duration_fallback")
"""
with open("/content/adaptive_threshold_verification.py", "w") as f:
    f.write(script)

!python /content/adaptive_threshold_verification.py

In [ ]:
# 15. Install resemble-enhance for the Tier 2 opt-in verification (not
# pinned in pyproject.toml -- Colab-only install, same precedent as
# soundfile/botocore). Confirmed effectively unmaintained upstream (gap.md
# evidence, 2026-11-20: last commit to main was Dec 3, 2024) but still used
# per explicit user decision since Tier 2 is never auto-selected.
!pip install -q resemble-enhance
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Diagnose the resemble-enhance install failure: find its declared pandas
# pin to see if it's requesting a version with no prebuilt wheel for this
# Colab Python version.
!pip download --no-deps --no-binary :all: -d /content/re_download resemble-enhance 2>&1 | tail -5
!python -c "import urllib.request; print(urllib.request.urlopen('https://pypi.org/pypi/resemble-enhance/json').read().decode())" > /content/re_meta.json
import json
meta = json.load(open("/content/re_meta.json"))
print(meta["info"]["requires_dist"])
print("latest version:", meta["info"]["version"])
import sys
print("python version:", sys.version)

In [ ]:
# resemble-enhance hard-pins exact package versions from late 2023
# (pandas==2.1.3, torch==2.1.1, numpy==1.26.2, ...) -- none of which have
# wheels for Colab's current Python 3.13, so a normal `pip install` can't
# even resolve. Try installing the package itself with --no-deps (skip its
# stale pin list) and let it run against whatever's already installed
# (torch/numpy/etc. are already present in this Colab image).
!pip install -q --no-deps resemble-enhance
import importlib
try:
    mod = importlib.import_module("resemble_enhance.enhancer.inference")
    print("Import OK:", mod)
except Exception as e:
    print(f"Import FAILED: {type(e).__name__}: {e}")

In [ ]:
!pip install -q deepspeed 2>&1 | tail -15
import importlib
try:
    mod = importlib.import_module("resemble_enhance.enhancer.inference")
    print("Import OK:", mod)
except Exception as e:
    print(f"Import FAILED: {type(e).__name__}: {e}")

In [ ]:
# 16. Set up Resemble Enhance's isolated legacy venv (gap.md Step 7 blocker
# fix: it can't coexist with audiobook_engine's numpy>=2.0 / Colab's Python
# 3.13). This runs apt-get + a full legacy pip install -- expect it to take
# a few minutes.
script = """
from pathlib import Path
from audiobook_engine.enhancement.resemble import ensure_resemble_enhance_env

venv_python = ensure_resemble_enhance_env(Path("/content/resemble_env"))
print(f"Venv python: {venv_python}")
"""
with open("/content/setup_resemble_env.py", "w") as f:
    f.write(script)

!python /content/setup_resemble_env.py

In [ ]:
# 17. Confirm the isolated venv actually fixes the import failure that
# blocked the previous in-process approach.
!/content/resemble_env/bin/python -c "from resemble_enhance.enhancer.inference import denoise, enhance; print('Import OK')"
!/content/resemble_env/bin/python -c "import torch; print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())"

In [ ]:
# 18. End-to-end mechanical verification of the isolated-venv Tier 2 path.
# This Colab runtime has no GPU attached (torch.cuda.is_available() is False
# in both the main kernel and the legacy venv -- a runtime-type setting, not
# a code issue), so call run_tier2_processing directly with device="cpu"
# rather than through run_job (whose dispatch currently always requests
# "cuda" -- device selection threading through run_job is a separate,
# later concern, not part of this increment's scope). yt-dlp is currently
# blocked by YouTube from this Colab IP (unrelated infra issue), so this
# uses a short synthetic tone instead of a real download -- still exercises
# the full real subprocess -> isolated venv -> resemble_enhance pipeline,
# just not real speech content. Mechanical check only (output produced,
# playable length) -- not a by-ear quality claim, which is the separate,
# still-deferred Urdu voice-preservation listening test.
script = """
from pathlib import Path
import numpy as np
import soundfile as sf

from audiobook_engine.enhancement.resemble import run_tier2_processing

OUTPUT_DIR = Path("/content/samples")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VENV_PYTHON = Path("/content/resemble_env/bin/python")

sr = 44100
t = np.linspace(0, 5.0, int(sr * 5.0), endpoint=False)
tone = (0.3 * np.sin(2 * np.pi * 220.0 * t)).astype(np.float32)
input_path = OUTPUT_DIR / "tier2_input_synthetic.wav"
sf.write(str(input_path), tone, sr)
print(f"Input: {input_path} -- {len(tone)/sr:.2f}s at {sr}Hz")

output_path = OUTPUT_DIR / "tier2_output.wav"
run_tier2_processing(input_path, output_path, sr, venv_python=VENV_PYTHON, device="cpu")

out_samples, out_sr = sf.read(str(output_path))
print(f"Tier 2 output: {output_path} -- {len(out_samples)/out_sr:.2f}s at {out_sr}Hz")
"""
with open("/content/tier2_smoke_test.py", "w") as f:
    f.write(script)

!python /content/tier2_smoke_test.py

In [ ]:
!/content/resemble_env/bin/python /content/audiobook-enhancer/audiobook_engine/enhancement/_resemble_worker.py --input /content/samples/tier2_input_synthetic.wav --output /content/samples/tier2_output.wav --device cpu

In [ ]:
# Resemble Enhance downloads its model checkpoints via `git lfs pull` on
# first use -- git-lfs isn't installed on this Colab image by default.
!apt-get install -y -qq git-lfs
!git lfs install
!/content/resemble_env/bin/python /content/audiobook-enhancer/audiobook_engine/enhancement/_resemble_worker.py --input /content/samples/tier2_input_synthetic.wav --output /content/samples/tier2_output.wav --device cpu

In [ ]:
import soundfile as sf
samples, sr = sf.read("/content/samples/tier2_output.wav")
print(f"Tier 2 output confirmed: {len(samples)/sr:.2f}s at {sr}Hz, {len(samples)} samples")

In [ ]:
# 19. Check that the already-stitched runner-smoke-001 job (gap.md Step 5
# evidence) still has its whole-file deliverable in Neon storage -- music
# mixing (Phase 5) depends only on "Phase 2 complete", so we can reuse this
# existing job's enhanced/final.wav rather than re-running the whole
# chunking/runner/stitching pipeline just to get a narration file to mix into.
from audiobook_engine.jobstore.storage import get_s3_client, BUCKET_NAME, object_key

client = get_s3_client()
key = object_key("runner-smoke-001", "enhanced", "final.wav")
try:
    head = client.head_object(Bucket=BUCKET_NAME, Key=key)
    print(f"Found {key}: {head['ContentLength']} bytes")
except Exception as e:
    print(f"NOT FOUND ({type(e).__name__}): {e}")

In [ ]:
# 20. End-to-end mechanical verification of music/mixing.py's mix_job
# against the real runner-smoke-001 narration. No real royalty-free music
# clip is available in this session, so this uses a synthetic tone as the
# music track (same precedent as the Tier 2 verification's synthetic input,
# gap.md Step 7) -- this checks the real I/O path (ffmpeg resample, Neon
# download/upload, mix_audio) actually runs end to end, not mix quality.
import numpy as np
import soundfile as sf
from pathlib import Path

work_dir = Path("/content/music_mix_work")
work_dir.mkdir(parents=True, exist_ok=True)

# A short synthetic "music" clip, deliberately a different sample rate
# (22050Hz) than the narration to exercise mix_job's ffmpeg-resample step.
music_sr = 22050
duration_s = 20.0
t = np.linspace(0, duration_s, int(music_sr * duration_s), endpoint=False)
tone = (0.3 * np.sin(2 * np.pi * 220.0 * t)).astype(np.float32)
music_path = work_dir / "synthetic_music.wav"
sf.write(str(music_path), tone, music_sr)
print(f"Synthetic music track written: {duration_s}s at {music_sr}Hz")

from audiobook_engine.music.mixing import mix_job

result_path = mix_job("runner-smoke-001", music_path, work_dir)
print(f"mix_job returned: {result_path}")

In [ ]:
# 21. Verify: local output duration matches the narration's, and the
# music_mixed artifact actually landed in Neon storage at the expected key.
mixed_samples, mixed_sr = sf.read(str(result_path))
narration_samples, narration_sr = sf.read(str(work_dir / "download_narration.wav"))
print(f"Narration: {len(narration_samples)/narration_sr:.2f}s at {narration_sr}Hz")
print(f"Mixed output: {len(mixed_samples)/mixed_sr:.2f}s at {mixed_sr}Hz")
assert len(mixed_samples) == len(narration_samples)
assert mixed_sr == narration_sr
assert float(np.max(np.abs(mixed_samples))) <= 1.0 + 1e-4
print("Duration/sample-rate match confirmed, no clipping beyond peak-safety clamp.")

mixed_key = object_key("runner-smoke-001", "music_mixed", "final.wav")
head = client.head_object(Bucket=BUCKET_NAME, Key=mixed_key)
print(f"Confirmed in Neon storage: {mixed_key} ({head['ContentLength']} bytes)")

In [ ]:
# 22. Verify the skip path for real too: mix_job with music_path=None must
# return None and upload nothing -- confirm the no-op passthrough actually
# holds against the real storage client, not just the local unit test.
skip_result = mix_job("runner-smoke-001", None, work_dir)
print(f"mix_job(music_path=None) returned: {skip_result!r}")
assert skip_result is None

# enhanced/final.wav (Phase 2's deliverable) must be untouched -- same size
# as before mixing ever ran.
enhanced_key = object_key("runner-smoke-001", "enhanced", "final.wav")
head = client.head_object(Bucket=BUCKET_NAME, Key=enhanced_key)
print(f"enhanced/final.wav unchanged: {head['ContentLength']} bytes")

In [ ]:
# 23. Load DATABASE_URL from Colab Secrets (same pattern as cell 3's S3
# credentials) -- needed for api/db.py's PostgresJobStore, which
# runner/poller.py uses to claim real queued rows from the real jobs table
# (api/migrations/001_create_jobs.sql, gap.md Step 9).
import os
from google.colab import userdata

os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")
print("DATABASE_URL loaded into environment.")

In [ ]:
# 24. End-to-end live verification of the Colab-side job poller (gap.md
# Step 9 follow-up -- nothing processed queued rows before this). Submit a
# real job via PostgresJobStore, then run exactly one poller iteration
# (runner/poller.py's process_one_job): claims it, downloads it for real
# (yt-dlp), runs it through run_job with the "gentle" preset's tier_max=0
# cap applied (config/presets.py, exercises the new preset-enforcement
# path), stitches it, and reports success back to Postgres.
from pathlib import Path
from api.db import PostgresJobStore
from api.models import EnhancementPreset
from audiobook_engine.runner.poller import process_one_job
from audiobook_engine.jobstore.storage import object_key as _object_key

job_store = PostgresJobStore()
job = job_store.insert_job(
    source_url="https://youtu.be/XN9CVvZFkts",
    preset=EnhancementPreset.GENTLE,
    add_music=False,
    idempotency_key="poller-smoke-001",
)
print(f"Submitted job: {job.id} status={job.status.value} preset={job.preset.value}")

handled = process_one_job(job_store, Path("/content/poller_work"))
result = job_store.get_job(job.id)
print(f"process_one_job handled a job: {handled}")
print(f"Job after processing: status={result.status.value} progress={result.progress} error={result.error}")

key = _object_key(str(job.id), "enhanced", "final.wav")
head = client.head_object(Bucket=BUCKET_NAME, Key=key)
print(f"Confirmed real deliverable under the job's own Postgres UUID: {key} ({head['ContentLength']} bytes)")

# Second real check: addMusic=true must fail fast with the specific
# not-yet-supported error (no music-source mechanism exists yet), not
# hang or crash the poller.
music_job = job_store.insert_job(
    source_url="https://youtu.be/XN9CVvZFkts",
    preset=EnhancementPreset.BALANCED,
    add_music=True,
    idempotency_key="poller-smoke-002-addmusic",
)
process_one_job(job_store, Path("/content/poller_work"))
music_result = job_store.get_job(music_job.id)
print(f"addMusic=true job: status={music_result.status.value} error={music_result.error}")

# Clean up both real test rows.
import psycopg

with psycopg.connect(os.environ["DATABASE_URL"]) as conn:
    with conn.cursor() as cur:
        cur.execute(
            "DELETE FROM jobs WHERE idempotency_key IN (%s, %s)",
            ("poller-smoke-001", "poller-smoke-002-addmusic"),
        )
        print("Test rows deleted:", cur.rowcount)
    conn.commit()